<img src="https://drive.google.com/uc?export=view&id=1wYSMgJtARFdvTt5g7E20mE4NmwUFUuog" width="200">

[![Gen AI Experiments](https://img.shields.io/badge/Gen%20AI%20Experiments-GenAI%20Bootcamp-blue?style=for-the-badge&logo=artificial-intelligence)](https://github.com/buildfastwithai/gen-ai-experiments)
[![Gen AI Experiments GitHub](https://img.shields.io/github/stars/buildfastwithai/gen-ai-experiments?style=for-the-badge&logo=github&color=gold)](http://github.com/buildfastwithai/gen-ai-experiments)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/PLACEHOLDER)

## Master Generative AI in 8 Weeks
**What You'll Learn:**
- Master cutting-edge AI tools & frameworks
- 6 weeks of hands-on, project-based learning
- Weekly live mentorship sessions
- Join Innovation Community

Learn by building. Get expert mentorship and work on real AI projects.
[Start Your Journey](https://www.buildfastwithai.com/genai-course)

## AG2 (formerly AutoGen) — Building Collaborative Multi-Agent AI Systems

AG2 is the community-maintained open-source framework for building
multi-agent AI systems in Python. Originally created as AutoGen by Microsoft
Research, AG2 is maintained by the original contributors and is the version
that receives active development, new features, and community support.

> **Note:** This notebook uses `ag2` (install: `pip install ag2`), which
> imports as `from autogen import ...`. The repo's earlier AutoGen notebooks
> use Microsoft's separate fork. AG2 v0.11+ is the recommended version for
> new projects.

Key components:
- **ConversableAgent** — The base agent class. Any agent that can send and receive messages.
- **AssistantAgent** — A pre-configured LLM-backed agent for task completion.
- **UserProxyAgent** — Represents a human or automated feedback provider; can execute code.
- **GroupChat + GroupChatManager** — Orchestrate three or more agents with LLM-driven turn selection.
- **ContextVariables** — Shared state dictionary visible to all agents across a conversation.
- **MCPClient** — Native Model Context Protocol client for connecting agents to MCP servers.

In [ ]:
%pip install -q "ag2[openai,mcp]"

In [ ]:
import os
import logging
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Suppress verbose AG2 internal logging for cleaner notebook output
logging.getLogger("autogen").setLevel(logging.WARNING)

### 1. Hello World — ConversableAgent

The simplest AG2 pattern: a single agent that responds to a message.
`ConversableAgent` is the base class for everything in AG2. Give it an
`llm_config` and a `system_message` and it is ready to work.

In [ ]:
from autogen import ConversableAgent, LLMConfig

llm_config = LLMConfig(api_type="openai", model="gpt-4o-mini")

assistant = ConversableAgent(
    name="assistant",
    system_message="You are a helpful AI assistant. Be concise — reply in two sentences maximum.",
    llm_config=llm_config,
)

reply = assistant.generate_reply(
    messages=[{"role": "user", "content": "What is a multi-agent system and why does it matter?"}]
)
print(reply)

### 2. Two-Agent Pattern — Writer and Critic

AG2's classic pattern: an **AssistantAgent** does the work and a
**UserProxyAgent** provides automated feedback. The conversation continues
until the termination condition is met — no human needed.

This is useful for self-correcting pipelines: the critic catches errors and
the writer revises until the output passes review.

In [ ]:
from autogen import AssistantAgent, UserProxyAgent, LLMConfig

llm_config = LLMConfig(api_type="openai", model="gpt-4o-mini")

writer = AssistantAgent(
    name="writer",
    system_message=(
        "You are a Python expert. Write clean, well-commented code. "
        "After your code is approved by the reviewer, reply with TERMINATE."
    ),
    llm_config=llm_config,
)

reviewer = UserProxyAgent(
    name="reviewer",
    human_input_mode="NEVER",           # Fully automated — no human input required
    is_termination_msg=lambda msg: "TERMINATE" in msg.get("content", ""),
    max_consecutive_auto_reply=4,
    code_execution_config=False,
    llm_config=llm_config,
    system_message=(
        "You are a strict Python code reviewer. Check for correctness, edge cases, "
        "and readability. If the code is correct and handles edge cases, reply APPROVE. "
        "The writer should then reply TERMINATE."
    ),
)

result = reviewer.initiate_chat(
    writer,
    message="Write a Python function that safely parses a JSON string and returns None on any error.",
)

### 3. Tool Use — Functions as Agent Capabilities

Register any Python function as a tool that an agent can call during a
conversation. AG2 handles the function schema generation, the LLM tool call,
execution, and feeding results back into the conversation automatically.

Here we build a simple stock analysis assistant that can look up prices
and calculate changes.

In [ ]:
from autogen import ConversableAgent, LLMConfig, register_function

llm_config = LLMConfig(api_type="openai", model="gpt-4o-mini")


def get_stock_price(ticker: str) -> str:
    """Get the current stock price for a given ticker symbol."""
    # Mock prices — replace with a real API (e.g., yfinance) in production
    prices = {
        "AAPL": 189.30,
        "GOOGL": 175.50,
        "MSFT": 415.20,
        "NVDA": 875.40,
        "META": 512.80,
    }
    price = prices.get(ticker.upper())
    if price is None:
        return f"Ticker '{ticker}' not found. Available: {', '.join(prices.keys())}"
    return f"${price:.2f}"


def calculate_percentage_change(old_price: float, new_price: float) -> str:
    """Calculate and describe the percentage change between two prices."""
    if old_price == 0:
        return "Cannot calculate percentage change from a zero price."
    change = ((new_price - old_price) / old_price) * 100
    direction = "up" if change > 0 else "down"
    return f"Price moved {direction} {abs(change):.2f}% (from ${old_price:.2f} to ${new_price:.2f})"


# The analyst agent describes tools to the LLM; the user agent executes them
analyst = ConversableAgent(
    name="stock_analyst",
    system_message=(
        "You are a stock market analyst. Use your tools to look up prices and "
        "calculate changes. Show your work step by step. Say TERMINATE when done."
    ),
    llm_config=llm_config,
)

user = ConversableAgent(
    name="user",
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "TERMINATE" in msg.get("content", ""),
    llm_config=False,  # No LLM — this agent just sends the initial message
)

# register_function: caller=analyst (adds tool schema to LLM), executor=user (runs the function)
register_function(
    get_stock_price,
    caller=analyst,
    executor=user,
    name="get_stock_price",
    description="Get the current stock price for a given ticker symbol.",
)
register_function(
    calculate_percentage_change,
    caller=analyst,
    executor=user,
    name="calculate_percentage_change",
    description="Calculate and describe the percentage change between two prices.",
)

result = user.initiate_chat(
    analyst,
    message=(
        "What are the current prices of AAPL and MSFT? "
        "If AAPL was $150 six months ago, what's the percentage change?"
    ),
)

### 4. GroupChat — Three Agents on a Shared Task

`GroupChat` lets multiple agents take turns on a shared conversation.
A `GroupChatManager` decides who speaks next — by default it uses the LLM
to choose the most appropriate agent based on context, or you can use
`"round_robin"` for a fixed rotation.

Here we build a three-agent content pipeline: planner → writer → editor.

In [ ]:
from autogen import AssistantAgent, GroupChat, GroupChatManager, LLMConfig

llm_config = LLMConfig(api_type="openai", model="gpt-4o-mini")

planner = AssistantAgent(
    name="planner",
    system_message=(
        "You are a content strategist. Given a topic, produce a concise outline: "
        "title, three section headings, two bullet points each. No prose yet."
    ),
    llm_config=llm_config,
)

writer = AssistantAgent(
    name="writer",
    system_message=(
        "You are a technical writer. Take the planner's outline and write "
        "a 200-word blog post in markdown. Follow the outline exactly."
    ),
    llm_config=llm_config,
)

editor = AssistantAgent(
    name="editor",
    system_message=(
        "You are a senior editor. Review the writer's post. Give one round "
        "of specific feedback on clarity and flow, then output the final "
        "polished version. End your message with FINISHED."
    ),
    llm_config=llm_config,
)

group_chat = GroupChat(
    agents=[planner, writer, editor],
    messages=[],
    max_round=6,
    speaker_selection_method="round_robin",  # Fixed order: planner → writer → editor
)

manager = GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config,
)

planner.initiate_chat(
    manager,
    message="Topic: Why multi-agent AI systems are the future of software development.",
)

### 5. Native MCP Client — Connecting Agents to External Tools

AG2 v0.11+ includes a native **Model Context Protocol (MCP)** client via
`ag2[mcp]`. MCP is an open standard that lets agents connect to any MCP
server — file systems, databases, APIs, web browsers, and more — without
custom integration code.

Here we connect an AG2 agent to a local MCP server using `MCPClient` and
give the agent access to its tools automatically.

> **What is MCP?** Model Context Protocol is a standard for exposing tools
> and resources to AI agents. MCP servers exist for dozens of services.
> AG2's native client means any MCP-compatible tool works with your agents
> out of the box.

In [ ]:
import asyncio
from autogen import ConversableAgent, LLMConfig
from autogen.mcp import MCPClient

llm_config = LLMConfig(api_type="openai", model="gpt-4o-mini")

# MCPClient connects to any MCP server via stdio or SSE transport.
# Here we use the filesystem MCP server (npx @modelcontextprotocol/server-filesystem)
# as an example. In Colab, replace with any MCP server available to you.
#
# For a quick local demo without a running MCP server, the cell below
# shows the pattern — swap `command` for a real server in production.

async def demo_mcp_agent():
    # Connect to an MCP server and load its tools automatically
    async with MCPClient(
        {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", "/tmp"]}
    ) as mcp_client:
        # Pull all tools from the MCP server as AG2-compatible tool definitions
        mcp_tools = await mcp_client.list_tools()

        agent = ConversableAgent(
            name="file_agent",
            system_message=(
                "You are a file management assistant. Use the available tools "
                "to help the user work with files. Say DONE when finished."
            ),
            llm_config=llm_config,
        )

        # Register every MCP tool on the agent in one call
        mcp_client.register_tools(agent, mcp_tools)

        user = ConversableAgent(
            name="user",
            human_input_mode="NEVER",
            is_termination_msg=lambda m: "DONE" in m.get("content", ""),
            llm_config=False,
        )

        await user.a_initiate_chat(
            agent,
            message="List what's in the /tmp directory and create a file called hello.txt with 'Hello from AG2!'",
        )

# Run in a Colab-compatible way
import nest_asyncio
nest_asyncio.apply()
asyncio.run(demo_mcp_agent())

## Learn More

- **AG2 documentation**: https://docs.ag2.ai/
- **AG2 GitHub**: https://github.com/ag2ai/ag2
- **Handoffs & orchestration patterns**: https://docs.ag2.ai/latest/docs/user-guide/advanced-concepts/orchestration/
- **MCP integration guide**: https://docs.ag2.ai/latest/docs/user-guide/advanced-concepts/tools/mcp/
- **Install**: `pip install ag2` or `pip install "ag2[openai,mcp]"`

> AG2 installs as `ag2` and imports as `from autogen import ...`.
> It is the community-maintained continuation of the original AutoGen project.